# AI가 이미지를 처리하는 방법

이미지 처리 가이드: https://developers.openai.com/api/docs/guides/images-vision?format=url

AI 모델이 이미지를 '보고' 이해하는 이 능력을 흔히 비전(Vision)이라고 부릅니다. 어떻게 AI에게 이미지를 전달할 수 있을까요?

### 1. 이미지를 입력하는 방법
| 입력 방식 | 설명 | 장점 | 단점 |
| :--- | :--- | :--- | :--- |
| **이미지 URL** | 웹상에 공개된 이미지의 완전한 주소(링크)를 제공 | 코드가 간결해지고 요청 데이터 크기가 작음 | 이미지가 호스팅된 서버가 응답하지 않거나, 비공개 이미지인 경우 분석 불가 |
| **Base64 인코딩** | 이미지 자체를 긴 텍스트(알파벳과 숫자)로 변환하여 직접 전송 | 외부 서버 의존 없이 로컬/보안 이미지를 즉시 전송 가능 | 데이터 용량이 커져 네트워크 전송 속도가 느려지고, 메모리 사용량이 증가함 |
| **파일 API 사용** | 서비스 제공자의 API를 통해 파일을 먼저 업로드하고, 반환된 '파일 ID'를 사용 | 대용량 파일이나 여러 번 재사용할 이미지 처리에 최적화됨 | 파일 업로드 후 분석이라는 두 단계의 과정을 거쳐야 하므로 아키텍처가 약간 복잡해짐 |

### 2. 이미지 입력 요구 사항 (2026.08.17 기준)
| 구분 | 세부 항목 | 요구 사항 및 설명 |
| :--- | :--- | :--- |
| **지원 파일 형식** | **지원 확장자** | • **PNG** (`.png`)<br>• **JPEG / JPG** (`.jpeg`, `.jpg`)<br>• **WEBP** (`.webp`)<br>• **GIF** (`.gif`, *정지된 비애니메이션 파일만 지원*) |
| **크기 및 수량 제한** | **총 페이로드 크기** | 단일 요청당 최대 **512MB** |
| | **개별 이미지 개수** | 단일 요청당 최대 **1,500장** |
| **품질 및 콘텐츠 요구 사항** | **가시성 (품질)** | 사람이 육안으로 명확히 알아볼 수 있을 정도로 선명할 것 |
| | **워터마크 및 로고** | 분석 정확도를 위해 워터마크나 로고가 없을 것 |
| | **안전성 (콘텐츠 정책)** | 선정적·유해 콘텐츠 등 플랫폼 정책 위반 요소가 없을 것 

# 인터넷에 있는 이미지 설명 요청하기

In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [5]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text", 
                "text": "이 이미지에 대해 설명해 주세요."
            },
            {
                "type": "image_url", 
                "image_url": 
                    {"url": "https://images.unsplash.com/photo-1736264335247-8ec5664c8328?q=80&w=1887&auto=format&fit=crop&ixlib=rb-4.0.3&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D",
                     "detail": "auto"}, # 숨겨진 기본 값
            },
        ],
    }
]

일반적인 텍스트 기반 API 호출에서는 content 필드에 단순 문자열을 입력하지만, 이미지와 텍스트를 함께 보낼 때는 객체(Object)들의 배열(Array) 형태로 작성해야 합니다.

이때 image_url 객체 내부에는 명시하지 않아도 기본적으로 작동하는 detail 파라미터가 존재합니다. 이 옵션은 AI가 이미지를 얼마나 자세히 분석할지를 결정하며, 다음과 같이 세 가지 모드로 설정할 수 있습니다.

1. "detail": "low" (저해상도 모드):
- AI가 이미지를 512x512 픽셀 수준으로 축소하여 전체적인 윤곽만 파악합니다.
- 처리 속도가 매우 빠르고, API 토큰(비용) 소모가 아주 적습니다.

2. "detail": "high" (고해상도 모드):
- 이미지를 여러 개의 타일(512x512 픽셀)로 분할하여 구석구석 정밀하게 분석합니다.
- 이미지 속의 작은 글씨(OCR), 복잡한 데이터 표, 디테일한 질감까지 정확하게 읽어낼 수 있습니다.

3. "detail": "auto" (기본값):
- 입력된 이미지의 원본 크기와 해상도에 따라 서버가 자동으로 low 또는 high 모드를 결정하여 적용합니다.

In [6]:
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=messages
)

print(response.choices[0].message.content)

이미지에는 밝은 금발의 여성이 주방에서 요리하는 모습이 담겨 있습니다. 여성은 검은색 민소매 상·하의를 입고 미소를 지으며, 파란색 그릇에 담긴 달걀을 거품기로 섞고 있습니다. 왼쪽 프라이팬에는 베이컨이나 고기 조각이 익고 있으며, 앞쪽에는 가스레인지가 보입니다. 어두운 주방 내부와 창밖의 풍경이 대비되어 있고, 사진은 따뜻하면서도 다소 빈티지한 색감과 강한 명암으로 촬영되었습니다.


# 내가 가진 이미지로 설명 요청하기

In [7]:
import base64

# 이미지를 인코딩 하는 함수
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(
            image_file.read()
        ).decode("utf-8")

컴퓨터에 저장된 이미지는 결국 무수히 많은 0과 1의 조합(이진 데이터)입니다. Base64 인코딩은 컴퓨터에 저장된 복잡한 이진 데이터(0과 1의 조합)를 우리가 읽을 수 있는 일반 텍스트(알파벳 대소문자 52개, 숫자 10개, 기호 +, /) 형태로 변환하는 기술입니다.

### Base64의 작동 원리: 0과 1을 텍스트로 바꾸는 방법
1) 데이터 묶기 (8비트 -> 24비트): 먼저 원본 이미지 데이터를 8비트(1바이트) 단위로 읽어 들입니다. 그리고 이 8비트 데이터 3개를 하나로 묶어 총 24비트의 덩어리를 만듭니다.

2) 다시 쪼개기 (24비트 -> 6비트): 묶어둔 24비트 덩어리를 이번에는 6비트씩 4개로 다시 쪼갭니다.

3) 문자와 매칭하기: 6비트는 $2^6$, 즉 0부터 63까지 총 64가지의 숫자를 표현할 수 있습니다. 미리 정해둔 Base64 문자표를 보고 이 숫자를 해당하는 문자로 1:1 변환합니다. (예: 0은 A, 26은 a, 63은 /)

4) 패딩 (Padding): 원본 데이터가 딱 3바이트씩 떨어지지 않고 남는 경우가 있습니다. 이때는 빈자리를 0으로 채우고, 텍스트 끝에 = 기호를 붙여 빈자리였음을 표시해 줍니다.

결과적으로 원본 이미지 파일은 iVBORw0KGgoAAAANSUhEUgAA... 와 같이 엄청나게 긴 알파벳과 숫자의 조합으로 변하게 됩니다.



### 왜 이미지 전송에 Base64를 사용할까?
AI 모델과 통신할 때 주로 사용하는 데이터 형식인 JSON이나 XML은 철저히 텍스트 기반 포맷입니다. 이곳에 0과 1로 이루어진 원본 이진 데이터를 그대로 밀어 넣으면 시스템이 이를 제어 문자나 잘못된 코드로 인식해 데이터가 깨지거나 오류가 발생합니다. 하지만 이미지를 Base64(안전한 텍스트)로 변환하면 텍스트 데이터의 일부로 온전히 인식되어 시스템 충돌 없이 안전하게 전송할 수 있습니다.



### Base64의 치명적인 단점: 데이터 크기 증가
Base64 인코딩은 만능이 아닙니다. 앞서 살펴본 원리에 따라 3바이트의 데이터를 4바이트의 텍스트로 변환하기 때문에, 변환을 거치고 나면 원본 이미지보다 용량이 약 33% 정도 증가하게 됩니다.

작은 아이콘이나 썸네일을 전송할 때는 이 정도의 용량 증가가 큰 문제가 되지 않습니다. 하지만 수 메가바이트(MB)에 달하는 고해상도 이미지를 Base64로 변환하면 데이터 크기가 기하급수적으로 커져 네트워크 전송 속도가 눈에 띄게 느려지고 서버의 메모리 자원도 크게 낭비됩니다.

In [8]:
image_path = "data/images/귀멸의칼날_젠이츠.png"

# 이미지를 Base64로 인코딩
base64_image = encode_image(image_path)

In [9]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "이 이미지에 대해 설명해주세요."
            },
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
            },
        ]
    }
]

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=messages
)


print(response.choices[0].message.content)

이미지는 일본 애니메이션풍의 캐릭터 일러스트입니다. 화면 중앙에는 **노란색과 주황색의 짧은 머리**를 가진 소년이 강렬한 표정으로 정면을 바라보고 있습니다. 그는 흰 삼각형 무늬가 반복된 **노란색 하오리**와 검은 제복을 입고 있으며, 양손으로 장식된 일본도를 단단히 쥐고 있습니다.

주변에는 번개처럼 갈라지는 눈부신 황금빛 에너지와 불꽃이 퍼져 있어, 빠른 검격이나 전투 직전의 긴장감이 강조됩니다. 전체적으로 노란색·주황색·검은색이 중심을 이루며, 역동적인 구도와 강한 명암 대비 덕분에 속도감과 위압적인 분위기가 느껴집니다. 캐릭터의 복장과 분위기는 《귀멸의 칼날》의 **아가츠마 젠이츠**를 연상시킵니다.


# 여러 이미지 비교 분석 요청하기

In [10]:
image01_base64 = encode_image("data/images/틀린그림찾기_1.png")
image02_base64 = encode_image("data/images/틀린그림찾기_2.png")

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "틀린 그림 찾기를 알고 있는가? 두 그림은 거의 비슷하지만 다른 부분이 존재한다. 어떤 부분이 다른지 설명해 주세요."
            },
            {
                "type": "image_url",
                "image_url": {"url": (f"data:image/jpeg;base64," f"{image01_base64}")}
            },
            {
                "type": "image_url",
                "image_url": {"url": (f"data:image/jpeg;base64," f"{image02_base64}")}
            },
        ]
    }
]

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=messages
)


print(response.choices[0].message.content)

다른 부분은 두 곳입니다.

1. **왼쪽 위 하늘**: 구름 아래에서 날고 있는 작은 새가 첫 번째 그림에만 있고, 두 번째 그림에는 없습니다.  
2. **그림 중앙 왼쪽**: 나무 아래, 곰 인형 왼쪽에 있는 벌이 첫 번째 그림에만 있고, 두 번째 그림에는 없습니다.


In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "틀린 그림 찾기를 알고 있는가? 두 그림은 거의 비슷하지만 다른 부분이 존재한다. 어떤 부분이 다른지 설명해 주세요."
            },
            {
                "type": "image_url",
                "image_url": {"url": (f"data:image/jpeg;base64," f"{image01_base64}"), "detail": "high"}
            },
            {
                "type": "image_url",
                "image_url": {"url": (f"data:image/jpeg;base64," f"{image02_base64}"), "detail": "high"}
            },
        ]
    }
]

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=messages
)


print(response.choices[0].message.content)

두 그림의 다른 부분은 다음 두 곳입니다.

1. **왼쪽 위쪽 하늘**: 첫 번째 그림에는 날아가는 새가 있지만, 두 번째 그림에는 없습니다.
2. **가운데 왼쪽, 나무 아래쪽**: 첫 번째 그림에는 벌 한 마리가 있지만, 두 번째 그림에는 없습니다.
